In [1]:
import cupy as cp

In [2]:
import torch

In [3]:
if torch.cuda.is_available():
    device_properties = torch.cuda.get_device_properties(0)
    print(f"Device Name: {device_properties.name}")
    print(f"Total Memory: {device_properties.total_memory / (1024**3):.2f} GB")
    print(f"CUDA Capability: {device_properties.major}.{device_properties.minor}")
else:
    print("CUDA is not available.")

Device Name: NVIDIA GeForce RTX 4070 Ti SUPER
Total Memory: 15.58 GB
CUDA Capability: 8.9


In [4]:
from minio import Minio
import os
import pandas as pd
import platform
import time

In [5]:
os.name

'posix'

In [6]:
print(platform.system(), platform.release())

Linux 6.8.0-60-generic


In [2]:
os.environ["SSL_CERT_FILE"] = 'all-data/public.crt'

In [3]:
def download_data_without_creds(data):
    client = Minio(
    '94.124.179.195:9000',
    secure=True
    )
    data = client.fget_object(
        'datasets',
        f'data/{data}',
        f'all-data/{data}'
    )

In [6]:
download_data_without_creds('OpenML-har-orig.csv')

In [7]:
download_data_without_creds('OpenML-usps-orig.csv')

In [11]:
# GaMAC Test

In [7]:
from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

In [8]:
def gamac_test(data, target_measures):
    measures = {"BR": Internal.BR, "OS": Internal.OS, "MCR": Internal.MCR, "SYM": Internal.SYM}
    used_measures = [measures[x] for x in target_measures.split(sep=',')]
    df = pd.read_csv(f'devops/comparison/test-data/{data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
        # classes = [int(c[1]) for c in classes]
    else:
        classes = None
    df = df.drop('class', errors='ignore', axis=1)
    print(f'used data: {data}')
    print(f'used measures: {used_measures}')
    result = Gamac(target_measures=tuple(used_measures)).run(table=df, text=None, image=None, classes=classes)
    print(f'optimal.model: {result.model}')
    print(f'clusters: {result.model.labels_}')
    print(f'F1 score: {result.f1_score}')

In [11]:
start = time.time()
gamac_test('OpenML-har-orig-1.csv', 'SYM')
print('Work time:', time.time() - start)

used data: OpenML-har-orig-1.csv
used measures: [<Internal.SYM: ('sym', <function sym at 0x7ca4b5123ba0>)>]


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 1.5896570682525635s, {'SYM': 1.6117034318692025e-07} ===
=== MEASURES 0.23360657691955566s, {'SYM': 0.0001626101858959197} ===
=== ALGO 4.253891229629517s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 7, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 32.98233246803284s, FailedRun, {'bandwidth': 0.8716760816450403, 'max_iter': 269, 'tol': 1.270881467770765e-05} ===
=== ALGO 3.187751293182373s, FailedRun, {'name': 'DBSCAN', 'eps': 1.0725549437973312, 'eps_sq': 1.1503741074640963, 'min_samples': 5} ===
=== ALGO 15.88028883934021s, FailedRun, {'min_cluster_size': 11, 'min_samples': 13} ===
=== MEASURES 0.23395848274230957s, {'SYM': 2.0362294288109722e-05} ===
=== ALGO 33.10476779937744s, SuccessRun, {'threshold': 0.14876376417587933, 'branching_factor': 28, 'n_clusters': 4} ===
=== MEASURES 0.22291040420532227s, {'SYM': 6.207895400592109e-05} ===
=== ALGO 0.2775247097015381s, SuccessRun, {'name': 'KMeans', 'n_clusters': 4, 'max_iter': 100, 'tol': 0.000

In [12]:
start = time.time()
gamac_test('OpenML-usps-orig-1.csv', 'OS')
print('Work time:', time.time() - start)

used data: OpenML-usps-orig-1.csv
used measures: [<Internal.OS: ('os', <function os at 0x7ca4b51554e0>)>]


/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.0015916824340820312s, {'OS': -14822.953654571467} ===
=== MEASURES 0.0011398792266845703s, {'OS': -145.647204113948} ===
=== ALGO 0.38855504989624023s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 4, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 25.60080099105835s, FailedRun, {'bandwidth': 0.15495746341465685, 'max_iter': 294, 'tol': 5.6522251359079294e-05} ===
=== ALGO 2.584648847579956s, FailedRun, {'name': 'DBSCAN', 'eps': 0.5323691803007514, 'eps_sq': 0.28341694413409396, 'min_samples': 12} ===
=== ALGO 13.882475137710571s, FailedRun, {'min_cluster_size': 12, 'min_samples': 14} ===
=== MEASURES 0.001583099365234375s, {'OS': -110.42869221313589} ===
=== ALGO 36.13066601753235s, SuccessRun, {'threshold': 0.8330638235975786, 'branching_factor': 38, 'n_clusters': 5} ===
=== MEASURES 0.0014832019805908203s, {'OS': -40.5569114250363} ===
=== ALGO 0.14072012901306152s, SuccessRun, {'name': 'KMeans', 'n_clusters': 11, 'max_iter': 100, 'tol': 0.0001,